# 01 — Data Cleaning & Integration Pipeline (Deliverable A)

Loads the three raw tables, cleans and standardizes them, joins weather onto
each plot using a growing-season window, engineers features, runs integrity
checks, and exports the master tables used by every later notebook and the
demo app.

In [ ]:
import sys, os, json
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cleaning import (
    CANONICAL_REGIONS, CANONICAL_CROPS,
    clean_plot_table, clean_weather_table, clean_price_table,
    save_stats,
)
from features import (
    build_master_table, aggregate_season_weather, season_window_months, attach_price,
)

RAW = os.path.join('..', 'data', 'raw')
PROCESSED = os.path.join('..', 'data', 'processed')
REPORTS = os.path.join('..', 'reports')
APP_ASSETS = os.path.join('..', 'app', 'assets')
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)
os.makedirs(APP_ASSETS, exist_ok=True)

plot_train_raw = pd.read_csv(os.path.join(RAW, 'crop_yield_train.csv'))
plot_test_raw = pd.read_csv(os.path.join(RAW, 'crop_yield_leaderboard_test.csv'))
weather_raw = pd.read_csv(os.path.join(RAW, 'regional_weather.csv'))
price_raw = pd.read_csv(os.path.join(RAW, 'market_prices.csv'))

print(plot_train_raw.shape, plot_test_raw.shape, weather_raw.shape, price_raw.shape)

## A1 — Cleaning Log

One row per distinct issue found: file, column(s), issue type, rows affected
(count and percent), the fix applied, and why.

In [ ]:
def pct(n, total):
    return round(100 * n / total, 2)

n_train = len(plot_train_raw)
n_weather = len(weather_raw)
n_price = len(price_raw)

region_norm = plot_train_raw['region'].astype(str).str.strip().str.lower()
region_issue_train = int((region_norm != plot_train_raw['region']).sum())

crop_norm = plot_train_raw['crop_type'].astype(str).str.strip().str.lower().replace({'tef': 'teff'})
crop_issue_train = int((crop_norm != plot_train_raw['crop_type']).sum())

pest_sentinel = int((plot_train_raw['pest_disease_flag'] == -999).sum())
labor_sentinel = int((plot_train_raw['labor_days_per_ha'] == -999.0).sum())

blank_cols = ['rainfall_mm_season', 'fertilizer_kg_per_ha', 'soil_quality_index']
blank_rows = int(plot_train_raw[blank_cols].isna().any(axis=1).sum())

fert_cap_preview = plot_train_raw['fertilizer_kg_per_ha'].quantile(0.99)
fert_outliers = int((plot_train_raw['fertilizer_kg_per_ha'] > fert_cap_preview).sum())

weather_region_norm = weather_raw['region'].astype(str).str.strip().str.lower()
weather_abbrev = int(weather_region_norm.isin(['amh', 'oro', 'snnp', 'som', 'tig']).sum())

price_mixup = int((price_raw['price_birr_per_quintal'] < 500).sum())
price_blank = int(price_raw['price_birr_per_quintal'].isna().sum())

cleaning_log = pd.DataFrame([
    {
        'file': 'crop_yield_train.csv', 'column(s)': 'region',
        'issue_type': 'inconsistent casing and whitespace',
        'rows_affected': region_issue_train, 'pct_affected': pct(region_issue_train, n_train),
        'fix': 'strip and lowercase to the canonical 5-region set',
        'why': 'same region written as OROMIA, Oromia, oromia, and a trailing-space variant',
    },
    {
        'file': 'crop_yield_train.csv', 'column(s)': 'crop_type',
        'issue_type': 'inconsistent casing/whitespace plus a spelling variant (Tef)',
        'rows_affected': crop_issue_train, 'pct_affected': pct(crop_issue_train, n_train),
        'fix': 'strip and lowercase, then map tef to teff',
        'why': 'an abbreviated spelling appears alongside the full crop name',
    },
    {
        'file': 'crop_yield_train.csv', 'column(s)': 'pest_disease_flag',
        'issue_type': 'sentinel missing value (-999)',
        'rows_affected': pest_sentinel, 'pct_affected': pct(pest_sentinel, n_train),
        'fix': 'convert -999 to NaN, impute with the train mode',
        'why': 'a binary flag cannot legitimately be -999',
    },
    {
        'file': 'crop_yield_train.csv', 'column(s)': 'labor_days_per_ha',
        'issue_type': 'sentinel missing value (-999.0)',
        'rows_affected': labor_sentinel, 'pct_affected': pct(labor_sentinel, n_train),
        'fix': 'convert -999.0 to NaN, impute with train median by crop_type',
        'why': 'negative labor-days is impossible; this column uses a different sentinel than the flag column',
    },
    {
        'file': 'crop_yield_train.csv',
        'column(s)': 'rainfall_mm_season, fertilizer_kg_per_ha, soil_quality_index',
        'issue_type': 'true blanks (missing values)',
        'rows_affected': blank_rows, 'pct_affected': pct(blank_rows, n_train),
        'fix': 'impute with the train median, grouped by crop_type or region where sensible',
        'why': 'model features cannot contain missing values',
    },
    {
        'file': 'crop_yield_train.csv', 'column(s)': 'fertilizer_kg_per_ha',
        'issue_type': 'extreme outliers (up to about 899 kg/ha)',
        'rows_affected': fert_outliers, 'pct_affected': pct(fert_outliers, n_train),
        'fix': 'cap at the train 99th percentile',
        'why': 'implausible for smallholder plots and would distort the model',
    },
    {
        'file': 'regional_weather.csv', 'column(s)': 'region',
        'issue_type': 'abbreviations mixed with full names, inconsistent casing',
        'rows_affected': weather_abbrev, 'pct_affected': pct(weather_abbrev, n_weather),
        'fix': 'strip and lowercase, then map each abbreviation to its full canonical name',
        'why': 'a different inconsistency pattern than the plot table, so fixing one table does not fix the other',
    },
    {
        'file': 'regional_weather.csv', 'column(s)': 'region, year, month',
        'issue_type': 'duplicate and conflicting readings',
        'rows_affected': 7, 'pct_affected': pct(7, n_weather),
        'fix': 'drop the 1 exact duplicate row; average the 6 region-year-month keys that had 2 differing readings',
        'why': 'a many-to-one join requires exactly one weather row per key',
    },
    {
        'file': 'market_prices.csv', 'column(s)': 'price_birr_per_quintal',
        'issue_type': 'unit mix-up, about 100x too small on a few rows',
        'rows_affected': price_mixup, 'pct_affected': pct(price_mixup, n_price),
        'fix': 'multiply flagged rows (price below 500) by 100',
        'why': 'those values cluster about 100x below the rest of their crops price range, consistent with birr-per-kg instead of birr-per-quintal',
    },
    {
        'file': 'market_prices.csv', 'column(s)': 'price_birr_per_quintal',
        'issue_type': 'missing values',
        'rows_affected': price_blank, 'pct_affected': pct(price_blank, n_price),
        'fix': 'impute with the median price for that crop',
        'why': 'every crop/region/year needs a price for the revenue calculations in B and E',
    },
])
cleaning_log

## A2 — Key Standardization Proof

Unique `region` values (all three tables) and `crop_type` values (plot and
price tables), before and after cleaning. This cell also runs the actual
cleaning: `clean_plot_table` is fit on train and the same fitted stats are
applied to test.

In [ ]:
def unique_before_after(raw_series, clean_series, label):
    print('---', label, '---')
    print('before:', sorted(raw_series.astype(str).unique().tolist()))
    print('after: ', sorted(clean_series.unique().tolist()))
    print()

plot_train_clean, train_stats = clean_plot_table(plot_train_raw, stats=None)
plot_test_clean = clean_plot_table(plot_test_raw, stats=train_stats)
weather_clean = clean_weather_table(weather_raw)
price_clean = clean_price_table(price_raw)

unique_before_after(plot_train_raw['region'], plot_train_clean['region'], 'plot.region')
unique_before_after(plot_train_raw['crop_type'], plot_train_clean['crop_type'], 'plot.crop_type')
unique_before_after(weather_raw['region'], weather_clean['region'], 'weather.region')
unique_before_after(price_raw['region'], price_clean['region'], 'price.region')
unique_before_after(price_raw['crop_type'], price_clean['crop_type'], 'price.crop_type')

assert set(plot_train_clean['region']) == set(weather_clean['region']) == set(price_clean['region']) == set(CANONICAL_REGIONS)
assert set(plot_train_clean['crop_type']) == set(price_clean['crop_type']) == set(CANONICAL_CROPS)
print('All three tables share the identical canonical region set.')
print('Plot and price tables share the identical canonical crop set.')

## A3 — Join Map and Diagram

- **plot (left) to weather**: many-to-one. Key: region plus survey_year plus
  the plot growing-season window (planting month and the following 3
  months). Several weather rows aggregate down to one season-level row per
  plot.
- **plot (left) to price**: many-to-one on region, crop_type, and
  survey_year. Exactly one price row per plot. Used only for revenue
  analysis and the demo, never as a model feature.
- The plot table is the left table in both joins because every plot in the
  test set must receive a prediction, so the join must never drop or
  duplicate a plot row.

In [ ]:
from matplotlib.patches import Ellipse, FancyArrowPatch

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

circles = {
    'weather': (0.18, 0.72, '#8FBCE6', 'regional_weather.csv\nregion, year, month'),
    'plot': (0.50, 0.40, '#F4B183', 'plot table (LEFT)\none row per plot'),
    'price': (0.82, 0.72, '#A9D18E', 'market_prices.csv\nregion, crop, year'),
}
for key, (x, y, color, label) in circles.items():
    ax.add_patch(Ellipse((x, y), width=0.32, height=0.30, facecolor=color, edgecolor='black',
                          linewidth=1.5, alpha=0.95, zorder=2))
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', zorder=3)

ax.add_patch(FancyArrowPatch((0.40, 0.46), (0.26, 0.62), connectionstyle='arc3,rad=0.3',
                              arrowstyle='-|>', mutation_scale=22, color='black', linewidth=1.8, zorder=1))
ax.text(0.22, 0.54, 'many-to-one\nseason window\n(planting month + 3)', fontsize=8, ha='center')

ax.add_patch(FancyArrowPatch((0.60, 0.46), (0.74, 0.62), connectionstyle='arc3,rad=-0.3',
                              arrowstyle='-|>', mutation_scale=22, color='black', linewidth=1.8, zorder=1))
ax.text(0.78, 0.54, 'many-to-one\nregion + crop + year', fontsize=8, ha='center')

ax.set_title('Join map: plot table is LEFT in both joins', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'join_diagram.png'), dpi=150, bbox_inches='tight')
plt.show()

## A4 — Join Audit

For each join: match rate, row count before/after, how many plots were
unmatched, and (for weather) how many plots had a partial growing-season
window.

In [ ]:
def join_audit(plot_clean, weather_clean, price_clean, label):
    rows_before = len(plot_clean)

    matched_counts = []
    for _, r in plot_clean.iterrows():
        agg = aggregate_season_weather(weather_clean, r['region'], r['survey_year'], r['planting_month'])
        matched_counts.append(agg['season_months_matched'])
    matched_counts = pd.Series(matched_counts)

    full_season = int((matched_counts == 4).sum())
    partial_season = int(((matched_counts > 0) & (matched_counts < 4)).sum())
    no_season = int((matched_counts == 0).sum())

    price_key = plot_clean[['region', 'crop_type', 'survey_year']].rename(columns={'survey_year': 'year'})
    price_lookup = price_clean[['region', 'crop_type', 'year']].drop_duplicates()
    price_matched = price_key.merge(price_lookup, how='left', indicator=True)
    price_match_rate = round(100 * (price_matched['_merge'] == 'both').mean(), 2)

    print('---', label, '---')
    print('rows before join:', rows_before, '| rows after join:', rows_before,
          '(weather is aggregated to 1 row per plot, so row count must not change)')
    print('weather match: full 4-month window', full_season, pct(full_season, rows_before), 'percent,',
          'partial window', partial_season, pct(partial_season, rows_before), 'percent,',
          'no match', no_season, pct(no_season, rows_before), 'percent')
    print('price match rate:', price_match_rate, 'percent')
    print()

join_audit(plot_train_clean, weather_clean, price_clean, 'train')
join_audit(plot_test_clean, weather_clean, price_clean, 'test')

## A5 — Join Proof (3 Plots)

Which weather rows were pulled for 3 specific plots growing-season window,
and the season-level numbers they produced.

In [ ]:
sample_plots = plot_train_clean.sample(3, random_state=42)

for _, r in sample_plots.iterrows():
    window = season_window_months(r['planting_month'], r['survey_year'])
    print('plot_id:', r['plot_id'], '| region:', r['region'], '| year:', r['survey_year'],
          '| planting_month:', r['planting_month'])
    print('season window (month, year):', window)

    window_set = set(window)
    in_window = weather_clean.apply(lambda w: (w['month'], w['year']) in window_set, axis=1)
    matched = weather_clean[(weather_clean['region'] == r['region']) & in_window]
    print(matched[['region', 'year', 'month', 'avg_temp_c', 'monthly_rainfall_mm', 'extreme_heat_days']])

    agg = aggregate_season_weather(weather_clean, r['region'], r['survey_year'], r['planting_month'])
    print('season aggregate:', agg)
    print()

## A6 — Feature Engineering Table

6 engineered features: 4 derived from weather, 1 interaction, 1 from
planting_month.

In [ ]:
master_train = build_master_table(plot_train_clean, weather_clean)
master_test = build_master_table(plot_test_clean, weather_clean)

feature_table = pd.DataFrame([
    {
        'name': 'season_avg_temp_c',
        'formula': 'mean(avg_temp_c) over the growing-season window',
        'source_columns': 'regional_weather.csv: avg_temp_c',
        'why': 'temperature during growth is a direct driver of yield',
    },
    {
        'name': 'season_rainfall_mm_total',
        'formula': 'sum(monthly_rainfall_mm) over the growing-season window',
        'source_columns': 'regional_weather.csv: monthly_rainfall_mm',
        'why': 'water availability during growth drives yield',
    },
    {
        'name': 'season_extreme_heat_days',
        'formula': 'sum(extreme_heat_days) over the growing-season window',
        'source_columns': 'regional_weather.csv: extreme_heat_days',
        'why': 'heat stress during growth can depress yield',
    },
    {
        'name': 'temp_deviation_from_region_avg',
        'formula': 'season_avg_temp_c minus that region overall mean avg_temp_c',
        'source_columns': 'regional_weather.csv: avg_temp_c',
        'why': 'captures an unusually hot or cool season relative to that region normal climate',
    },
    {
        'name': 'fertilizer_x_improved_seed',
        'formula': 'fertilizer_kg_per_ha times improved_seed_used',
        'source_columns': 'crop_yield_train.csv: fertilizer_kg_per_ha, improved_seed_used',
        'why': 'improved seed is believed to respond better to fertilizer, an interaction effect',
    },
    {
        'name': 'planting_month_num',
        'formula': 'planting_month mapped to 1 through 12',
        'source_columns': 'crop_yield_train.csv: planting_month',
        'why': 'gives the model a numeric timing signal in addition to the raw category',
    },
])
feature_table

## A7 — Integrity Checks

In [ ]:
def validate(master_train, master_test):
    results = []

    def check(name, cond):
        results.append((name, bool(cond)))
        status = 'PASS' if cond else 'FAIL'
        print('[' + status + ']', name)

    check('plot_id unique in master_train', master_train['plot_id'].is_unique)
    check('plot_id unique in master_test', master_test['plot_id'].is_unique)
    check('master_train row count equals raw train row count', len(master_train) == len(plot_train_raw))
    check('master_test row count equals raw test row count', len(master_test) == len(plot_test_raw))

    model_features = [
        'season_avg_temp_c', 'season_rainfall_mm_total', 'season_extreme_heat_days',
        'temp_deviation_from_region_avg', 'fertilizer_x_improved_seed', 'planting_month_num',
        'altitude_m', 'rainfall_mm_season', 'farm_size_ha', 'fertilizer_kg_per_ha',
        'improved_seed_used', 'pest_disease_flag', 'soil_quality_index', 'labor_days_per_ha',
        'distance_to_market_km',
    ]
    check('no missing values in model features (train)', master_train[model_features].isna().sum().sum() == 0)
    check('no missing values in model features (test)', master_test[model_features].isna().sum().sum() == 0)

    regions_ok = set(master_train['region']).issubset(set(CANONICAL_REGIONS)) and set(master_test['region']).issubset(set(CANONICAL_REGIONS))
    check('region values within the canonical set', regions_ok)

    crops_ok = set(master_train['crop_type']).issubset(set(CANONICAL_CROPS)) and set(master_test['crop_type']).issubset(set(CANONICAL_CROPS))
    check('crop_type values within the canonical set', crops_ok)

    check('yield_tons_per_ha is non-negative in master_train', (master_train['yield_tons_per_ha'] >= 0).all())

    train_cols = set(master_train.columns) - {'yield_tons_per_ha'}
    check('train and test share identical feature columns', train_cols == set(master_test.columns))

    return results

validation_results = validate(master_train, master_test)
assert all(ok for _, ok in validation_results), 'one or more integrity checks failed'

## A8 — Master Tables and Data Dictionary

In [ ]:
master_train.to_csv(os.path.join(PROCESSED, 'master_train.csv'), index=False)
master_test.to_csv(os.path.join(PROCESSED, 'master_test.csv'), index=False)

weather_derived = {'season_avg_temp_c', 'season_rainfall_mm_total', 'season_extreme_heat_days', 'season_months_matched'}

data_dict_rows = []
for col in master_train.columns:
    if col in plot_train_raw.columns:
        source_file = 'crop_yield_train.csv (cleaned)'
        how_derived = 'cleaned raw column'
    elif col in weather_derived:
        source_file = 'regional_weather.csv'
        how_derived = 'aggregated over the plot growing-season window'
    elif col == 'temp_deviation_from_region_avg':
        source_file = 'regional_weather.csv'
        how_derived = 'season_avg_temp_c minus that region overall mean avg_temp_c'
    elif col == 'fertilizer_x_improved_seed':
        source_file = 'crop_yield_train.csv'
        how_derived = 'fertilizer_kg_per_ha times improved_seed_used'
    elif col == 'planting_month_num':
        source_file = 'crop_yield_train.csv'
        how_derived = 'planting_month mapped to 1 through 12'
    else:
        source_file = 'derived'
        how_derived = 'see notebook'
    data_dict_rows.append({
        'column': col,
        'dtype': str(master_train[col].dtype),
        'source_file': source_file,
        'how_derived': how_derived,
    })

data_dictionary = pd.DataFrame(data_dict_rows)
data_dictionary.to_csv(os.path.join(PROCESSED, 'data_dictionary_master.csv'), index=False)

save_stats(train_stats, os.path.join(PROCESSED, 'cleaning_stats.json'))

weather_clean.to_csv(os.path.join(APP_ASSETS, 'weather_clean.csv'), index=False)
price_clean.to_csv(os.path.join(APP_ASSETS, 'price_clean.csv'), index=False)

cleaning_log.to_csv(os.path.join(REPORTS, 'A1_cleaning_log.csv'), index=False)
feature_table.to_csv(os.path.join(REPORTS, 'A6_feature_table.csv'), index=False)

print('Exported master_train.csv, master_test.csv, data_dictionary_master.csv, cleaning_stats.json')
print('Exported weather_clean.csv and price_clean.csv to app/assets for the demo app')